In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

train = pd.read_csv('../data/raw/train.csv', low_memory=False)
store = pd.read_csv('../data/raw/store.csv')

df = train.merge(store, on='Store', how='left')
print("Shape:", df.shape)
df.head()

Shape: (1017209, 18)


,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,31-07-2015,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,31-07-2015,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,31-07-2015,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,31-07-2015,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,31-07-2015,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [18]:
print(f"Before filter: {len(df):,} rows")
df = df[df['Open'] == 1]
df = df[df['Sales'] > 0]
print(f"After filter: {len(df):,} rows")
print(f"Removed: {1017209 - len(df):,} rows")

Before filter: 1,017,209 rows
After filter: 844,338 rows
Removed: 172,871 rows


In [19]:
# StateHoliday has both integer 0 and string "0" - standardize to string
df['StateHoliday'] = df['StateHoliday'].astype(str)
print(df['StateHoliday'].value_counts())

StateHoliday
0    843428
a       694
b       145
c        71
Name: count, dtype: int64


In [20]:
print(f"CompetitionDistance nulls before: {df['CompetitionDistance'].isnull().sum()}")
df['CompetitionDistance'] = df['CompetitionDistance'].fillna(999999)
print(f"CompetitionDistance nulls after: {df['CompetitionDistance'].isnull().sum()}")

CompetitionDistance nulls before: 2186
CompetitionDistance nulls after: 0


In [21]:
# How many months has the competitor been open?
df['CompetitionOpenSinceYear'] = df['CompetitionOpenSinceYear'].fillna(0)
df['CompetitionOpenSinceMonth'] = df['CompetitionOpenSinceMonth'].fillna(0)

df['Date'] = pd.to_datetime(df['Date'])

df['CompetitionOpenMonths'] = (
    (df['Date'].dt.year - df['CompetitionOpenSinceYear']) * 12 +
    (df['Date'].dt.month - df['CompetitionOpenSinceMonth'])
)
df['CompetitionOpenMonths'] = df['CompetitionOpenMonths'].clip(lower=0)
print(df['CompetitionOpenMonths'].describe())

count    844338.000000
mean       7731.474911
std       11229.439988
min           0.000000
25%          29.000000
50%          91.000000
75%       24163.000000
max       24187.000000
Name: CompetitionOpenMonths, dtype: float64


C:\Users\rushi\AppData\Local\Temp\ipykernel_29256\1090671436.py:5: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df['Date'] = pd.to_datetime(df['Date'])


In [22]:
df['DayOfWeek'] = df['Date'].dt.dayofweek        # 0=Monday, 6=Sunday
df['Month'] = df['Date'].dt.month
df['Year'] = df['Date'].dt.year
df['WeekOfYear'] = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfMonth'] = df['Date'].dt.day
df['IsWeekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['DaysToMonthEnd'] = df['Date'].dt.days_in_month - df['Date'].dt.day

print(df[['DayOfWeek','Month','Year','WeekOfYear','IsWeekend','DaysToMonthEnd']].head())

   DayOfWeek  Month  Year  WeekOfYear  IsWeekend  DaysToMonthEnd
0          4      7  2015          31          0               0
1          4      7  2015          31          0               0
2          4      7  2015          31          0               0
3          4      7  2015          31          0               0
4          4      7  2015          31          0               0


In [23]:
promo_month_map = {
    'Jan':1, 'Feb':2, 'Mar':3, 'Apr':4,
    'May':5, 'Jun':6, 'Jul':7, 'Aug':8,
    'Sep':9, 'Sept':9, 'Oct':10, 'Nov':11, 'Dec':12
}

def is_promo2_active(row):
    if row['Promo2'] == 0:
        return 0
    if pd.isnull(row['PromoInterval']):
        return 0
    try:
        active_months = [promo_month_map[m.strip()] for m in row['PromoInterval'].split(',')]
        current_month = row['Date'].month
        return 1 if current_month in active_months else 0
    except KeyError:
        return 0

df['Promo2Active'] = df.apply(is_promo2_active, axis=1)
print(df['Promo2Active'].value_counts())

Promo2Active
0    699125
1    145213
Name: count, dtype: int64


In [24]:
# StoreType and Assortment — label encode
df['StoreType'] = df['StoreType'].map({'a':0, 'b':1, 'c':2, 'd':3})
df['Assortment'] = df['Assortment'].map({'a':0, 'b':1, 'c':2})
df['StateHoliday'] = df['StateHoliday'].map({'0':0, 'a':1, 'b':2, 'c':3})

print(df[['StoreType','Assortment','StateHoliday']].head())

   StoreType  Assortment  StateHoliday
0          2           0             0
1          0           0             0
2          0           0             0
3          2           2             0
4          0           0             0


In [25]:
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

for lag in [7, 14, 21, 28]:
    df[f'sales_lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)

print(df[[f'sales_lag_{lag}' for lag in [7,14,21,28]]].isnull().sum())

sales_lag_7      7805
sales_lag_14    15610
sales_lag_21    23415
sales_lag_28    31220
dtype: int64


In [26]:
for window in [7, 14, 28]:
    df[f'sales_rolling_mean_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(window).mean())
    )
    df[f'sales_rolling_std_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(window).std())
    )

print("Rolling features created")
print(df[[f'sales_rolling_mean_{w}' for w in [7,14,28]]].isnull().sum())

Rolling features created
sales_rolling_mean_7      7805
sales_rolling_mean_14    15610
sales_rolling_mean_28    31220
dtype: int64


In [30]:
print(f"Before dropna: {len(df):,} rows")

df = df.dropna(subset=['sales_lag_28', 'sales_rolling_mean_28']).reset_index(drop=True)

print(f"After dropna: {len(df):,} rows")
print(f"Remaining nulls:\n{df.isnull().sum()[df.isnull().sum() > 0]}")

# Drop leaky columns — errors='ignore' skips already-dropped columns
df = df.drop(columns=['Customers', 'Open', 'PromoInterval',
                       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
                       'Promo2SinceWeek', 'Promo2SinceYear'], errors='ignore')

# Resave
df.to_parquet('../data/processed/features.parquet', index=False)
print(f"\nFinal shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Before dropna: 405,058 rows
After dropna: 405,058 rows
Remaining nulls:
Series([], dtype: int64)

Final shape: (405058, 29)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenMonths', 'Month', 'Year', 'WeekOfYear', 'DayOfMonth', 'IsWeekend', 'DaysToMonthEnd', 'Promo2Active', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_14', 'sales_rolling_std_14', 'sales_rolling_mean_28', 'sales_rolling_std_28']


In [32]:
print(f"Before dropna: {len(df):,} rows")

df = df.dropna(subset=['sales_lag_28', 'sales_rolling_mean_28']).reset_index(drop=True)

print(f"After dropna: {len(df):,} rows")

# Drop leaky columns — errors='ignore' skips already-dropped columns
df = df.drop(columns=['Customers', 'Open', 'PromoInterval',
                       'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
                       'Promo2SinceWeek', 'Promo2SinceYear'], errors='ignore')

# Resave
df.to_parquet('../data/processed/features.parquet', index=False)
print(f"\nFinal shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Before dropna: 405,058 rows
After dropna: 405,058 rows

Final shape: (405058, 29)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenMonths', 'Month', 'Year', 'WeekOfYear', 'DayOfMonth', 'IsWeekend', 'DaysToMonthEnd', 'Promo2Active', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_14', 'sales_rolling_std_14', 'sales_rolling_mean_28', 'sales_rolling_std_28']


In [34]:
# Interaction features
df['Promo_DayOfWeek'] = df['Promo'] * df['DayOfWeek']
df['StoreType_Promo'] = df['StoreType'] * df['Promo']

print("Interaction features added")
print(df[['Promo_DayOfWeek', 'StoreType_Promo']].describe())

Interaction features added
       Promo_DayOfWeek  StoreType_Promo
count    405058.000000    405058.000000
mean          0.882301         0.597359
std           1.358762         1.142947
min           0.000000         0.000000
25%           0.000000         0.000000
50%           0.000000         0.000000
75%           2.000000         0.000000
max           4.000000         3.000000


In [35]:
df.to_parquet('../data/processed/features.parquet', index=False)
print(f"Final shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

Final shape: (405058, 31)
Columns: ['Store', 'DayOfWeek', 'Date', 'Sales', 'Promo', 'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'Promo2', 'CompetitionOpenMonths', 'Month', 'Year', 'WeekOfYear', 'DayOfMonth', 'IsWeekend', 'DaysToMonthEnd', 'Promo2Active', 'sales_lag_7', 'sales_lag_14', 'sales_lag_21', 'sales_lag_28', 'sales_rolling_mean_7', 'sales_rolling_std_7', 'sales_rolling_mean_14', 'sales_rolling_std_14', 'sales_rolling_mean_28', 'sales_rolling_std_28', 'Promo_DayOfWeek', 'StoreType_Promo']
